# Notebook 3 — Anomaly Detection
**Purpose:** Find unusual financial patterns using Z-score and Isolation Forest methods.
Detect anomalous years per company across sales, net_profit, borrowings, operating_profit.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, sqlite3, os, sys
from sklearn.ensemble import IsolationForest
from scipy import stats
sys.path.insert(0, os.path.abspath('..'))
sns.set_theme(style='darkgrid'); plt.rcParams['figure.figsize'] = (14, 6)

conn = sqlite3.connect(os.path.join('..', 'db.sqlite3'))
years = pd.read_sql('SELECT * FROM dim_year ORDER BY sort_order', conn)
companies = pd.read_sql('SELECT c.*, s.sector_name FROM dim_company c LEFT JOIN dim_sector s ON c.sector_id=s.sector_id', conn)
pl = pd.read_sql('SELECT * FROM fact_profit_loss', conn).merge(years[['year_id','year_label','sort_order','fiscal_year']], on='year_id')
bs = pd.read_sql('SELECT * FROM fact_balance_sheet', conn).merge(years[['year_id','year_label','sort_order']], on='year_id')
print(f'P&L rows: {len(pl)}, BS rows: {len(bs)}')

## Step 1: Z-Score Anomaly Detection

In [ ]:
METRICS = ['sales', 'net_profit', 'operating_profit']
z_threshold = 2.5

anomalies_z = []
for symbol in pl['company_id'].unique():
    comp_pl = pl[pl['company_id'] == symbol].sort_values('sort_order')
    for metric in METRICS:
        vals = comp_pl[metric].dropna()
        if len(vals) < 4: continue
        z_scores = np.abs(stats.zscore(vals))
        anomaly_idx = vals.index[z_scores > z_threshold]
        for idx in anomaly_idx:
            row = comp_pl.loc[idx]
            anomalies_z.append({
                'company_id': symbol, 'year_label': row['year_label'],
                'metric': metric, 'value': row[metric],
                'z_score': z_scores[vals.index.get_loc(idx)], 'method': 'Z-Score'
            })

# Borrowings from balance sheet
for symbol in bs['company_id'].unique():
    comp_bs = bs[bs['company_id'] == symbol].sort_values('sort_order')
    vals = comp_bs['borrowings'].dropna()
    if len(vals) < 4: continue
    z_scores = np.abs(stats.zscore(vals))
    for idx in vals.index[z_scores > z_threshold]:
        row = comp_bs.loc[idx]
        anomalies_z.append({
            'company_id': symbol, 'year_label': row['year_label'],
            'metric': 'borrowings', 'value': row['borrowings'],
            'z_score': z_scores[vals.index.get_loc(idx)], 'method': 'Z-Score'
        })

z_df = pd.DataFrame(anomalies_z)
print(f'Z-Score anomalies detected: {len(z_df)}')
z_df.sort_values('z_score', ascending=False).head(15)

## Step 2: Isolation Forest Anomaly Detection

In [ ]:
# Build feature matrix per company-year
feat = pl[['company_id','year_label','sort_order','sales','net_profit','operating_profit','opm_pct','eps']].copy()
feat = feat.merge(bs[['company_id','year_id','borrowings','debt_to_equity','total_assets']].rename(columns={'year_id':'year_id_bs'}),
                  left_on=['company_id'], right_on=['company_id'], how='left')

# Aggregate to company-year level
feat_cols = ['sales','net_profit','operating_profit','opm_pct','eps','borrowings','debt_to_equity']
feat_clean = feat.groupby(['company_id','year_label'])[feat_cols].first().reset_index()
feat_matrix = feat_clean[feat_cols].fillna(0)

iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
feat_clean['iso_anomaly'] = iso_forest.fit_predict(feat_matrix)
feat_clean['iso_score'] = iso_forest.decision_function(feat_matrix)

iso_anomalies = feat_clean[feat_clean['iso_anomaly'] == -1]
print(f'Isolation Forest anomalies: {len(iso_anomalies)}')
iso_anomalies.sort_values('iso_score').head(15)

## Step 3: Visualize Anomalies on Timeline

In [ ]:
# Companies with most anomalous years
if len(z_df) > 0:
    top_anomaly_cos = z_df['company_id'].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(top_anomaly_cos.index, top_anomaly_cos.values, color='#EF4444')
    ax.set_title('Companies with Most Z-Score Anomalies', fontweight='bold')
    ax.invert_yaxis(); plt.tight_layout(); plt.show()

# Timeline scatter: anomalies by year
if len(z_df) > 0:
    fig, ax = plt.subplots(figsize=(16, 6))
    year_counts = z_df['year_label'].value_counts().sort_index()
    ax.bar(year_counts.index, year_counts.values, color='#F59E0B', edgecolor='white')
    ax.set_title('Anomaly Count by Year', fontweight='bold')
    plt.xticks(rotation=45, fontsize=8); plt.tight_layout(); plt.show()

## Step 4: Cross-Reference with Known Events

In [ ]:
known_events = {
    'COVID-2020': {'years': ['Mar 2020','Mar 2021'], 'sectors': ['Aviation','Real Estate','Consumer']},
    'Adani-2023': {'companies': ['ADANIENT','ADANIPORTS','ADANIPOWER'], 'years': ['Mar 2023']},
    'IT-Boom-2021': {'sectors': ['IT'], 'years': ['Mar 2021','Mar 2022']},
}

print('=== Cross-Reference: Anomalies vs Known Events ===\n')
for event, details in known_events.items():
    mask = z_df['year_label'].isin(details.get('years', []))
    if 'companies' in details:
        mask &= z_df['company_id'].isin(details['companies'])
    matches = z_df[mask]
    print(f'{event}: {len(matches)} anomalies found')
    if len(matches) > 0:
        print(matches[['company_id','year_label','metric','value','z_score']].to_string(index=False))
    print()

## Step 5: Compare Z-Score vs Isolation Forest

In [ ]:
# Overlap analysis
z_keys = set(zip(z_df['company_id'], z_df['year_label'])) if len(z_df) > 0 else set()
iso_keys = set(zip(iso_anomalies['company_id'], iso_anomalies['year_label']))
both = z_keys & iso_keys
print(f'Z-Score only: {len(z_keys - iso_keys)}')
print(f'Isolation Forest only: {len(iso_keys - z_keys)}')
print(f'Both methods agree: {len(both)}')

# Venn-like visualization
fig, ax = plt.subplots(figsize=(8, 5))
try:
    from matplotlib_venn import venn2
    venn2([z_keys, iso_keys], set_labels=('Z-Score', 'Isolation Forest'), ax=ax)
except ImportError:
    ax.bar(['Z-Score Only', 'Both', 'IsoForest Only'],
           [len(z_keys-iso_keys), len(both), len(iso_keys-z_keys)],
           color=['#3B82F6','#10B981','#F59E0B'])
ax.set_title('Anomaly Detection Method Comparison', fontweight='bold')
plt.tight_layout(); plt.show()

## Step 6: Export Anomaly Flags

In [ ]:
# Combine both methods
all_anomalies = []
if len(z_df) > 0:
    for _, r in z_df.iterrows():
        all_anomalies.append({'company_id': r['company_id'], 'year_label': r['year_label'],
                              'metric': r['metric'], 'method': 'Z-Score', 'score': r['z_score']})
for _, r in iso_anomalies.iterrows():
    all_anomalies.append({'company_id': r['company_id'], 'year_label': r['year_label'],
                          'metric': 'multivariate', 'method': 'IsolationForest', 'score': abs(r['iso_score'])})

anomaly_export = pd.DataFrame(all_anomalies)
anomaly_export.to_csv('../data/anomaly_flags.csv', index=False)
print(f'✅ Exported {len(anomaly_export)} anomaly flags to data/anomaly_flags.csv')
conn.close()